# ノートブック 04b：人のメッシュ動画を正面から描き直す（ランタイム：T4 で可）

ノートブック1で保存した GVHMR の推定結果 `hmr4d_results.pt` を読み込み、**推定はやり直さずに**、カメラの位置だけを変えて描き直します。
元の `2_global.mp4` は「最初のフレームの体の向き」から 45° の位置にカメラが固定されていて、踊り手の平均の向きとずれていたため背中側からの画になっていました。
`render_gvhmr_front.py` は全フレームの体の向きを平均し、その正面にカメラを置きます（床・照明・レンズは元と同じ）。

**事前準備**：手元の `pai_final/scripts/render_gvhmr_front.py` をドライブの `pai_final/scripts/` にアップロード。
所要時間の目安：インストール 15〜25 分＋描画 10〜20 分。出力は `$P/outputs/gvhmr/the_feels_full/2_global_front.mp4`（元の動画は残ります）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
P = '/content/drive/MyDrive/pai_final'
os.environ['P'] = P

# 使うリポジトリのコミット（この版で動作を確認したソースに固定。切断後に入れ直しても同じ版になる）
PINS = {
    'GVHMR':            ('https://github.com/zju3dv/GVHMR',                  'ee960bb6e2ea2d381aa97f08e9b71ef320b624b1'),
    'GMR':              ('https://github.com/YanjieZe/GMR',                  'bb1bbe40774794fceb2a7c579a3464a28e68c844'),
    'unitree_rl_mjlab': ('https://github.com/unitreerobotics/unitree_rl_mjlab', '1425b15f73bd4095f0df53709d7c389c3eb9e790'),
    'mjlab':            ('https://github.com/mujocolab/mjlab',               '5af32e378dcb93c9e881ace83cc5a3f5d373fe60'),  # v1.2.0
}
for k, (url, rev) in PINS.items():
    os.environ[f'URL_{k}'], os.environ[f'REV_{k}'] = url, rev

# サビ後半 20 秒が、全編（伊原六花さんの動画 2分19秒）の何秒目か（音声の照合で特定済み）
SEG_START, SEG_END = 59.815, 80.78
os.environ['SEG_START'], os.environ['SEG_END'] = str(SEG_START), str(SEG_END)

# MuJoCo を画面なし（EGL）で描画するための設定
NVIDIA_ICD = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD):
    os.makedirs(os.path.dirname(NVIDIA_ICD), exist_ok=True)
    with open(NVIDIA_ICD, 'w') as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYTHONUNBUFFERED'] = '1'   # ログをすぐファイルに書き出す

# 動画・画像をノートブック上で表示する関数
from base64 import b64encode
from IPython.display import HTML, Image, display
def show(path, width=720):
    data = b64encode(open(path, 'rb').read()).decode()
    return HTML(f'<video width={width} controls loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

os.environ['FULL'] = 'the_feels_full'   # 全編（2分19秒）
os.environ['SEG'] = 'the_feels_20s'     # サビ後半 20 秒

!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls -la $P/input $P/scripts
!ls -R $P/models | head -20

## 1. GVHMR のインストール（ノートブック1の 2-2 と同じ）

In [ ]:
%%bash
set -e
pip -q install uv
cd /content
bash $P/scripts/pin_clone.sh $URL_GVHMR /content/GVHMR $REV_GVHMR
uv venv /content/venv_gvhmr --python 3.10 --seed -q
PY=/content/venv_gvhmr/bin/python
# ビルド時に numpy / pip を必要とするパッケージを先に入れる
$PY -m pip -q install "numpy==1.23.5" cython "setuptools>=68" wheel
$PY -m pip -q install --no-build-isolation chumpy cython_bbox
cd GVHMR
$PY -m pip -q install -r requirements.txt
$PY -m pip -q install -e .
$PY -c "import torch, pytorch3d; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())"

## 2. 人体モデルを配置（推定はしないので、学習済みモデルのコピーは省略）

In [ ]:
%%bash
set -e
cd /content/GVHMR
mkdir -p inputs/checkpoints/body_models
cp -r $P/models/smpl $P/models/smplx inputs/checkpoints/body_models/
ls $P/scripts/render_gvhmr_front.py
ls -la $P/outputs/gvhmr/$FULL/hmr4d_results.pt $P/input/$FULL.mp4

## 3. 試しに最初の 3 秒だけ描く（向きの確認）

顔と胸がこちらを向いていれば OK。逆なら `--view_deg 180`、斜め前にしたいなら `--view_deg 30` などに変えて描き直します。

In [ ]:
%%bash
set -e
cd /content/GVHMR
/content/venv_gvhmr/bin/python $P/scripts/render_gvhmr_front.py --pred $P/outputs/gvhmr/$FULL/hmr4d_results.pt \
  --video $P/input/$FULL.mp4 --out /content/front_test.mp4 --view_deg 0 --max_frames 90

In [ ]:
display(show("/content/front_test.mp4", width=640))

## 4. 全編を描く

In [ ]:
%%bash
set -e
cd /content/GVHMR
/content/venv_gvhmr/bin/python $P/scripts/render_gvhmr_front.py --pred $P/outputs/gvhmr/$FULL/hmr4d_results.pt \
  --video $P/input/$FULL.mp4 --out $P/outputs/gvhmr/$FULL/2_global_front.mp4 --view_deg 0
ls -la $P/outputs/gvhmr/$FULL/

In [ ]:
display(show(f"{P}/outputs/gvhmr/{os.environ['FULL']}/2_global_front.mp4", width=640))